# 05 — Add a New Stock


## 1 — Bootstrap


In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
SRC = ROOT / 'src'
assert SRC.exists(), f'Could not find src/ at {SRC}'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

import config
from collect import (collect_prices, registry_template, verify_all_raw,
                     verify_raw_file)
from pipeline import TrainConfig, artifacts_exist, predict_latest, train_stock
from validate import run_all_checks

config.ensure_dirs()
pd.set_option('display.width', 140)

print(f'Registered stocks: {config.list_stocks()}')


Registered stocks: ['ADANIENT', 'ADANIGREEN', 'ETERNAL', 'HDFCBANK', 'INFY', 'ITC', 'RELIANCE', 'SBI', 'TCS', 'VEDL']


## 2 — Health check


In [3]:
health = run_all_checks(deep=False)


  PROJECT HEALTH CHECK

Dependencies
------------
  [ok] import numpy             core
  [ok] import pandas            core
  [ok] import sklearn           core
  [ok] import joblib            core
  [ok] import ta                indicators
  [ok] import matplotlib        plots
  [ok] import yfinance          downloading new price data
  [ok] import torch             LSTM / GRU sequence models
  [ok] import streamlit         the dashboard
  [ok] import plotly            the dashboard
  [ok] import transformers      FinBERT sentiment
  [ok] import vaderSentiment    VADER sentiment
  [ok] import requests          news collection

Project modules
---------------
  [ok] src/config.py            imports cleanly
  [ok] src/dataio.py            imports cleanly
  [ok] src/features.py          imports cleanly
  [ok] src/dataset.py           imports cleanly
  [ok] src/stationary.py        imports cleanly
  [ok] src/targets.py           imports cleanly
  [ok] src/evaluation.py        imports clea

KeyboardInterrupt: 

## 3 — Define the stock to add


In [4]:
NEW_KEY = 'ADANIGREEN'
NEW_TICKER = 'ADANIGREEN.NS'
NEW_NAME = 'Adani Green Energy'
NEW_START = '2014-01-01'
NEW_END = '2024-01-01'

print(registry_template(
    NEW_KEY,
    NEW_TICKER,
    NEW_NAME,
    start=NEW_START,
    end=NEW_END
))

    "ADANIGREEN": StockConfig(
        key="ADANIGREEN",
        ticker="ADANIGREEN.NS",
        display_name="Adani Green Energy",
        raw_filename="ADANIGREEN_raw.csv",
        start="2014-01-01",
        end="2024-01-01",
        sentiment_filename=None,
        notes="Added 2026-08-30.",
    ),


## 4 — Paste the block above into `src/config.py`, then restart the kernel


In [5]:
import importlib

import config
importlib.reload(config)

print(f'Registered stocks: {config.list_stocks()}')
assert NEW_KEY in config.STOCKS, (
    f'{NEW_KEY} is not in config.STOCKS. Paste the block from step 3 into '
    f'src/config.py, save the file, then restart the kernel and re-run.')
print(f'{NEW_KEY} found in registry')


Registered stocks: ['ADANIENT', 'ADANIGREEN', 'ETERNAL', 'HDFCBANK', 'INFY', 'ITC', 'RELIANCE', 'SBI', 'TCS', 'VEDL']
ADANIGREEN found in registry


## 5 — Download price history


In [ ]:
path = collect_prices(NEW_KEY, overwrite=True)
print()
result = verify_raw_file(NEW_KEY)


  ETERNAL: downloading ETERNAL.NS 2021-01-01 .. 2026-09-01
  ETERNAL: saved 1265 rows -> data/raw/ETERNAL_raw.csv

  ETERNAL      OK  1265 rows, 2021-07-23 .. 2026-08-28 (5.1y)


## 6 — Confirm the history is long enough


In [120]:
if not result.get('ok'):
    raise RuntimeError(f"Download failed: {result.get('problem')}")

if result.get('warning'):
    print('WARNING')
    print(f"  {result['warning']}")
    print()
    print('  The pipeline will still train, at a shorter horizon.')
    print('  For a horizon-3 model, re-run step 5 with an earlier start date.')
else:
    print(f"{NEW_KEY}: {result['rows']} rows over {result['years']} years.")
    print('Sufficient for a horizon-3 model.')


ETERNAL: 1265 rows over 5.1 years.
Sufficient for a horizon-3 model.


## 7 — Build the dataset


In [121]:
from dataset import build_dataset
from features import audit_dataset, print_audit
from stationary import add_stationary_features, get_stationary_features

df = build_dataset(NEW_KEY, with_sentiment=False, save=True, verbose=True)
df = add_stationary_features(df).dropna().reset_index(drop=True)
features = get_stationary_features(df)

print_audit(audit_dataset(df, features), title=f'Audit: {NEW_KEY}')



=== Building dataset: ETERNAL (Eternal Limited) ===
  loaded ETERNAL_raw.csv: 1265 rows (0 dropped), 2021-07-23 to 2026-08-28
  features: 1265 -> 1215 rows (50 dropped as warm-up/unlabelled), 35 columns

Audit: ETERNAL
--------------
  rows                 : 1215
  columns              : 35
  leaky_features       : None
  has_inf              : False
  nan_counts           : None
  dates_sorted         : True
  duplicate_dates      : 0
  date_range           : 2021-10-04 to 2026-08-27
  target_balance       : {1: 0.5021, 0: 0.4979}
  majority_baseline    : 0.5021
  sentiment_merged     : False
  n_features           : 20
  saved ETERNAL_dataset.csv: 1215 rows x 35 cols -> processed/

Audit: ETERNAL
--------------
  rows                 : 1147
  columns              : 77
  leaky_features       : None
  has_inf              : False
  nan_counts           : None
  dates_sorted         : True
  duplicate_dates      : 0
  date_range           : 2021-12-30 to 2026-08-27
  target_balance    

## 8 — Train and save artifacts


In [122]:
cfg = TrainConfig()   # defaults: logistic, horizon 3, non-overlapping

artifacts = train_stock(NEW_KEY, cfg=cfg, verbose=True)



=== Training ETERNAL (Eternal Limited) ===
  full frame     : 1147 rows, 52 features
  modelling frame: 376 rows (horizon=3, k=0.0)
  walk-forward   : 0.5591 +/- 0.0540 (baseline 0.5080, edge +0.0511)
  seed-averaged  : 0.5591 (edge +0.0511) -> POSITIVE (offsets untested)
  permutation    : NO SIGNAL (shuffled 0.4954)
  operating point: coverage 60%, accuracy 0.5664, edge +0.0619
  abstention     : ADOPTED (edge improves +0.0000 -> +0.0619 at 60% coverage)
  saved -> models/ETERNAL/


## 9 — Verify the trained model


In [123]:
meta = artifacts['metadata']
perm = meta['permutation'] or {}
seeds = meta['seed_robustness'] or {}

summary = pd.Series({
    'stock': meta['stock_key'],
    'rows': meta['n_modelling_rows'],
    'features': meta['n_features'],
    'horizon': meta['config']['horizon'],
    'accuracy': meta['walk_forward_accuracy'],
    'std': meta['walk_forward_std'],
    'baseline': meta['baseline'],
    'edge': meta['edge'],
    'permutation': perm.get('verdict'),
    'seed_verdict': seeds.get('verdict'),
})
print(summary.to_string())
print()

if perm.get('verdict') == 'LEAKAGE':
    print('STOP: permutation test indicates leakage. Do not use this model.')
elif meta['edge'] <= 0:
    print('No demonstrated edge. The model is valid but has no skill on this')
    print('stock. The dashboard will show a warning banner. This is a normal')
    print('and reportable outcome, not a bug.')
else:
    print('Model has a positive, permutation-clean edge.')


stock                               ETERNAL
rows                                    376
features                                 52
horizon                                   3
accuracy                             0.5591
std                                   0.054
baseline                              0.508
edge                                 0.0511
permutation                       NO SIGNAL
seed_verdict    POSITIVE (offsets untested)

Model has a positive, permutation-clean edge.


## 10 — Offset robustness


In [124]:
from modeling import evaluate_across_offsets
from targets import build_directional_dataset

horizon = meta['config']['horizon']

if horizon > 1:
    def builder(frame, offset):
        return build_directional_dataset(
            frame.iloc[offset:].reset_index(drop=True),
            horizon=horizon, k=meta['config']['k'], non_overlapping=True)

    offs, edges = evaluate_across_offsets(
        df, features, meta['config']['model_name'],
        dataset_builder=builder, n_offsets=horizon,
        n_splits=meta['config']['n_splits'],
        embargo=meta['config']['embargo'])

    off_df = pd.DataFrame({'offset': offs, 'edge': [round(e, 4) for e in edges]})
    print(off_df.to_string(index=False))
    print()
    if all(e > 0 for e in edges):
        print('Positive at every sampling offset. The edge is robust.')
    else:
        print('Edge flips negative at one or more offsets.')
        print('Report the mean across offsets, not the best one.')
else:
    print('Horizon is 1; there is only one sampling offset to test.')


 offset    edge
      0  0.0511
      1  0.0060
      2 -0.0058

Edge flips negative at one or more offsets.
Report the mean across offsets, not the best one.


## 11 — Final check


In [125]:
print(predict_latest(NEW_KEY))
print()

rows = []
for key in config.list_stocks():
    rows.append({'stock': key, 'trained': artifacts_exist(key)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print('Launch the dashboard with:   streamlit run app.py')


{'stock': 'ETERNAL', 'date': '2026-08-27', 'close': 328.5, 'probability_up': 0.6168, 'confidence': 0.1168, 'confidence_threshold': 0.0965, 'signal': 'UP', 'horizon_days': 3, 'expected_accuracy': 0.5664, 'expected_coverage': 0.6011}

     stock  trained
  ADANIENT     True
ADANIGREEN     True
   ETERNAL     True
  HDFCBANK     True
      INFY     True
       ITC     True
  RELIANCE     True
       SBI     True
       TCS     True
      VEDL     True

Launch the dashboard with:   streamlit run app.py


## 12 — Rebuild the comparison report


In [126]:
import json
from pipeline import load_artifacts

rows = []
for key in config.list_stocks():
    if not artifacts_exist(key):
        continue
    m = load_artifacts(key)['metadata']
    rows.append({
        'stock': key,
        'horizon': m['config']['horizon'],
        'model': m['config']['model_name'],
        'rows': m['n_modelling_rows'],
        'accuracy': m['walk_forward_accuracy'],
        'std': m['walk_forward_std'],
        'baseline': m['baseline'],
        'edge': m['edge'],
        'permutation': (m['permutation'] or {}).get('verdict'),
    })

table = pd.DataFrame(rows).set_index('stock')
table.to_csv(config.REPORTS_DIR / 'all_stocks_summary.csv')
print(table.to_string())


            horizon     model  rows  accuracy     std  baseline    edge permutation
stock                                                                              
ADANIENT          3  logistic   995    0.5282  0.0407    0.5343 -0.0061      SIGNAL
ADANIGREEN        3  logistic   620    0.4742  0.0450    0.4968 -0.0226   NO SIGNAL
ETERNAL           3  logistic   376    0.5591  0.0540    0.5080  0.0511   NO SIGNAL
HDFCBANK          3  logistic   998    0.4873  0.0560    0.5739 -0.0866   NO SIGNAL
INFY              3  logistic   999    0.5100  0.0292    0.5220 -0.0120   NO SIGNAL
ITC               3  logistic   995    0.5523  0.0472    0.4941  0.0582      SIGNAL
RELIANCE          3  logistic   342    0.4383  0.0694    0.4916 -0.0533   NO SIGNAL
SBI               3  logistic   995    0.4860  0.0391    0.5518 -0.0658   NO SIGNAL
TCS               3  logistic   999    0.4840  0.0654    0.4840  0.0000   NO SIGNAL
VEDL              3  logistic   778    0.4776  0.0717    0.5582 -0.0806   NO

In [14]:
from modeling import evaluate_across_offsets
from targets import build_directional_dataset
from pipeline import load_artifacts
from dataset import build_dataset
from stationary import add_stationary_features, get_stationary_features
import pandas as pd
import config


def check_offsets(stock_key):
    print("=" * 60)
    print(f"OFFSET ROBUSTNESS: {stock_key}")
    print("=" * 60)

    # Load trained model metadata
    artifacts = load_artifacts(stock_key)
    meta = artifacts["metadata"]

    horizon = meta["config"]["horizon"]
    model_name = meta["config"]["model_name"]
    k = meta["config"]["k"]

    # Rebuild the same feature frame used by the pipeline
    df = build_dataset(
        stock_key,
        with_sentiment=False,
        save=False,
        verbose=False
    )

    df = add_stationary_features(df).dropna().reset_index(drop=True)
    features = get_stationary_features(df)

    if horizon <= 1:
        print("Horizon is 1 — only one sampling offset exists.")
        return

    def builder(frame, offset):
        return build_directional_dataset(
            frame.iloc[offset:].reset_index(drop=True),
            horizon=horizon,
            k=k,
            non_overlapping=True
        )

    offsets, edges = evaluate_across_offsets(
        df,
        features,
        model_name,
        dataset_builder=builder,
        n_offsets=horizon,
        n_splits=meta["config"]["n_splits"],
        embargo=meta["config"]["embargo"]
    )

    results = pd.DataFrame({
        "offset": offsets,
        "edge": edges
    })

    results["edge_pct"] = results["edge"] * 100

    print()
    print(results.round(4).to_string(index=False))

    print()
    print(f"Mean edge: {results['edge'].mean() * 100:.2f}%")
    print(f"Minimum edge: {results['edge'].min() * 100:.2f}%")
    print(f"Maximum edge: {results['edge'].max() * 100:.2f}%")

    if all(results["edge"] > 0):
        print()
        print("ROBUST: positive edge at every offset.")
    else:
        print()
        print("NOT FULLY ROBUST: edge becomes negative at one or more offsets.")

    return results


eternal_offsets = check_offsets("ETERNAL")
tcs_offsets = check_offsets("TCS")

OFFSET ROBUSTNESS: ETERNAL

 offset   edge  edge_pct
      0 0.1242   12.4242
      1 0.0070    0.7005
      2 0.0369    3.6932

Mean edge: 5.61%
Minimum edge: 0.70%
Maximum edge: 12.42%

ROBUST: positive edge at every offset.
OFFSET ROBUSTNESS: TCS

 offset   edge  edge_pct
      0 0.0154    1.5385
      1 0.0333    3.3333
      2 0.0103    1.0256

Mean edge: 1.97%
Minimum edge: 1.03%
Maximum edge: 3.33%

ROBUST: positive edge at every offset.
